In [45]:
from pathlib import Path
import pandas as pd
import numpy as np
from joblib import Parallel, delayed
from collections import defaultdict

In [46]:
ROOT = Path().cwd()
DATA_INITIAL = Path("../data_initial").resolve()
DATA = Path("../data").resolve()
FANTOM = DATA / "fantom_cat"
PREPROCESSING = Path("../preprocessing")
IPS = DATA_INITIAL / "deg/rnaseq_res_ips"
FIBRO = DATA_INITIAL / "deg/rnaseq_res_fibro"
IPS_INDEX = PREPROCESSING / "deg_index/ips_deg_index"
FIBRO_INDEX = PREPROCESSING / "deg_index/fibro_deg_index"
IPS_INDEX_V2 = PREPROCESSING / "deg_index-v2/ips_deg_index"
FIBRO_INDEX_V2 = PREPROCESSING / "deg_index-v2/fibro_deg_index"
BIOMART = DATA / "biomart"
BIOMART_INITIAL = DATA_INITIAL / "biomart"
IPS_PREPROC = PREPROCESSING / "deg/fibro"
FIBRO_PREPROC = PREPROCESSING / "deg/ips"

In [47]:
to_ips = IPS_PREPROC / "ips_df_with_deg_target_ens_genes_updated.tsv"
ips_expr_filt = pd.read_csv(to_ips, sep="\t")
ips_expr_filt.head()

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,ENSG00000205531,NAP1L4,-0.691972,3.899614e-06,0.042455,-5.061144,G0234787_08,ENSG00000234787,LINC00458,down
1,ENSG00000089916,GPATCH2L,-1.041246,3.913478e-05,0.043336,-3.953852,R0057746_03,CATG00000070943,CATG00000070943,down
2,ENSG00000099194,SCD,-0.635000,2.054724e-07,0.000870,-2.709444,R0057746_03,CATG00000070943,CATG00000070943,down
3,ENSG00000113161,HMGCR,-0.912822,1.601495e-05,0.029059,-3.682696,R0057746_03,CATG00000070943,CATG00000070943,down
4,ENSG00000128829,EIF2AK4,-0.887877,2.397361e-07,0.000870,-4.709886,R0057746_03,CATG00000070943,CATG00000070943,down


In [48]:
ips_expr_filt["target.new-ens_gene"].nunique()

114

In [49]:
ips_expr_filt["target.aso"].nunique()

246

In [50]:
ips_expr_filt[ips_expr_filt["target.new-gene_name"] == "EMX2OS"]

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr


In [51]:
ips_expr_filt.shape

(14743, 10)

In [52]:
ips_expr_filt["DEG.new-ens_gene"].nunique()

5359

In [53]:
to_fibro = FIBRO_PREPROC / "fibro_df_with_deg_target_ens_genes_updated.tsv"
fibro_expr_filt = pd.read_csv(to_fibro, sep="\t")
fibro_expr_filt.head()

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,ENSG00000200169,RNU5D-1,-2.325717,1.640034e-11,2.548449e-07,-3.106205,G0268895_01,ENSG00000268895,A1BG-AS1,down
1,ENSG00000138675,FGF5,-1.371879,7.814666e-11,6.071605e-07,-2.756217,G0268895_01,ENSG00000268895,A1BG-AS1,down
2,ENSG00000252521,RNU5D-2P,-1.925674,1.102746e-09,5.711854e-06,-2.534245,G0268895_01,ENSG00000268895,A1BG-AS1,down
3,ENSG00000268895,A1BG-AS1,-1.542280,3.509193e-09,1.363234e-05,-7.161737,G0268895_01,ENSG00000268895,A1BG-AS1,down
4,CATG00000098831,CATG00000098831,1.153985,1.174760e-08,3.480974e-05,2.301412,G0268895_01,ENSG00000268895,A1BG-AS1,up


In [54]:
fibro_expr_filt.shape

(128114, 10)

In [55]:
fibro_expr_filt["DEG.new-ens_gene"].nunique()

16340

In [56]:
fibro_expr_filt["target.new-ens_gene"].nunique()

148

In [57]:
fibro_expr_filt["target.aso"].nunique()

291

In [58]:
deg_results = {"ips":   ips_expr_filt,
               "fibro": fibro_expr_filt}

In [59]:
deg_results["ips"]

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,ENSG00000205531,NAP1L4,-0.691972,3.899614e-06,0.042455,-5.061144,G0234787_08,ENSG00000234787,LINC00458,down
1,ENSG00000089916,GPATCH2L,-1.041246,3.913478e-05,0.043336,-3.953852,R0057746_03,CATG00000070943,CATG00000070943,down
2,ENSG00000099194,SCD,-0.635000,2.054724e-07,0.000870,-2.709444,R0057746_03,CATG00000070943,CATG00000070943,down
3,ENSG00000113161,HMGCR,-0.912822,1.601495e-05,0.029059,-3.682696,R0057746_03,CATG00000070943,CATG00000070943,down
4,ENSG00000128829,EIF2AK4,-0.887877,2.397361e-07,0.000870,-4.709886,R0057746_03,CATG00000070943,CATG00000070943,down
...,...,...,...,...,...,...,...,...,...,...
14738,ENSG00000259132,ENSG00000259132,0.951549,3.207729e-05,0.002283,3.083371,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,up
14739,ENSG00000259203,ENSG00000259203,1.364151,6.251776e-05,0.003601,1.693698,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,up
14740,ENSG00000262481,TMEM256-PLSCR3,-0.742945,4.097390e-04,0.013477,-2.084815,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,down
14741,ENSG00000265565,MIR3143,-0.673589,3.604504e-05,0.002453,-3.019324,G0230844_AD_07,ENSG00000230844,ZNF674-AS1,down


In [60]:
deg_results["ips"]["target.new-ens_gene"].nunique(), deg_results["ips"]["target.aso"].nunique()

(114, 246)

In [61]:
deg_results["fibro"]

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,ENSG00000200169,RNU5D-1,-2.325717,1.640034e-11,2.548449e-07,-3.106205,G0268895_01,ENSG00000268895,A1BG-AS1,down
1,ENSG00000138675,FGF5,-1.371879,7.814666e-11,6.071605e-07,-2.756217,G0268895_01,ENSG00000268895,A1BG-AS1,down
2,ENSG00000252521,RNU5D-2P,-1.925674,1.102746e-09,5.711854e-06,-2.534245,G0268895_01,ENSG00000268895,A1BG-AS1,down
3,ENSG00000268895,A1BG-AS1,-1.542280,3.509193e-09,1.363234e-05,-7.161737,G0268895_01,ENSG00000268895,A1BG-AS1,down
4,CATG00000098831,CATG00000098831,1.153985,1.174760e-08,3.480974e-05,2.301412,G0268895_01,ENSG00000268895,A1BG-AS1,up
...,...,...,...,...,...,...,...,...,...,...
128109,ENSG00000232187,FTH1P7,2.554486,3.241567e-05,4.609803e-02,5.237118,G0267886_AD_04,ENSG00000183850,ZNF730,up
128110,ENSG00000142961,MOB3C,0.749422,6.254070e-06,4.815008e-02,3.582150,G0269293_01,ENSG00000269293,ZSCAN16-AS1,up
128111,ENSG00000132613,MTSS2,-0.535628,1.170372e-06,1.853869e-02,-1.732862,G0269293_AD_10,ENSG00000269293,ZSCAN16-AS1,down
128112,ENSG00000264395,MIR3193,1.530984,4.610931e-06,3.651857e-02,2.419552,G0269293_AD_10,ENSG00000269293,ZSCAN16-AS1,up


In [62]:
deg_results["fibro"]["target.new-ens_gene"].nunique(), deg_results["fibro"]["target.aso"].nunique()

(148, 291)

In [32]:
def indexing_deg(lnc_ens, aso, up_or_down_expr, subdf, project_dir):
    to_index = Path(project_dir) / lnc_ens
    to_index.mkdir(exist_ok=True, parents=True)
    to_save = to_index / f"{lnc_ens}-{aso}-{up_or_down_expr}-.tsv.gz"
    if to_save.exists():
        print(f"Skipped: {lnc_ens} {aso} {up_or_down_expr} {project_dir.stem}")
        return None
    subdf.to_csv(to_save, sep="\t", compression="gzip", index=False)    

In [33]:
with Parallel(n_jobs=-1) as pool:
        chunks = pool(
            delayed(indexing_deg)(lnc_ens, aso, up_or_down_expr, subdf, project_dir=PREPROCESSING / f"deg_index-v2/{exp}_deg_index")
            for exp in deg_results
            for (lnc_ens, aso, up_or_down_expr), subdf in deg_results[exp].groupby(by=["target.new-ens_gene", "target.aso", "up_or_down_expr"])
        )

In [63]:
#способ смотреть без разбиаения на up/down экспрессию
pd.concat([pd.read_csv(to_deg, sep="\t") for to_deg in IPS_INDEX_V2.glob((f'*/*ENSG00000142396-G0142396_AD_01-*'))]).head()

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,CATG00000118091,CATG00000118091,1.034982,4.409901e-05,0.013717,2.478310,G0142396_AD_01,ENSG00000142396,ERVK3-1,up
1,ENSG00000018625,ATP1A2,0.707376,1.901322e-05,0.007941,2.287828,G0142396_AD_01,ENSG00000142396,ERVK3-1,up
2,ENSG00000077238,IL4R,1.125659,4.566045e-05,0.013808,2.154211,G0142396_AD_01,ENSG00000142396,ERVK3-1,up
3,ENSG00000107731,UNC5B,0.997251,7.561714e-05,0.021664,2.281174,G0142396_AD_01,ENSG00000142396,ERVK3-1,up
4,ENSG00000109519,GRPEL1,0.764083,1.097960e-07,0.000239,3.790012,G0142396_AD_01,ENSG00000142396,ERVK3-1,up


In [64]:
#способ смотреть без разбиаения на up/down экспрессию
pd.concat([pd.read_csv(to_deg, sep="\t") for to_deg in FIBRO_INDEX_V2.glob((f'*/*ENSG00000243926-G0240875_10-*'))]).head()

,DEG.new-ens_gene,DEG.new-gene_name,log2FC,pvalue,fdr,ZScore,target.aso,target.new-ens_gene,target.new-gene_name,up_or_down_expr
0,ENSG00000111144,LTA4H,0.643336,5.975901e-10,0.000002,2.405249,G0240875_10,ENSG00000243926,TIPARP-AS1,up
1,ENSG00000197958,RPL12,1.634219,3.039196e-06,0.002936,2.625141,G0240875_10,ENSG00000243926,TIPARP-AS1,up
2,ENSG00000165072,MAMDC2,0.786023,3.527888e-06,0.003208,2.177767,G0240875_10,ENSG00000243926,TIPARP-AS1,up
3,ENSG00000137509,PRCP,0.528597,4.065014e-06,0.003411,2.395984,G0240875_10,ENSG00000243926,TIPARP-AS1,up
4,ENSG00000104435,STMN2,0.519063,1.043377e-05,0.006452,1.930233,G0240875_10,ENSG00000243926,TIPARP-AS1,up
